# Семинар 1. Изображение как массив данных — версия для студентов

**Курс:** Анализ изображений  

### Ритм семинара
В нескольких местах встретятся четыре типа мини-заданий:

- 🔮 **Предскажи** — сформулируй результат до запуска кода.
- 📷 **Проверь на своём фото** — используй фотографию с телефона.

На семинаре разберём `H × W × C`, RGB/BGR, `uint8` и `float`, crop/resize, grayscale, гистограммы, gamma и JPEG.


## 0. Импорты и вспомогательная функция

In [ ]:
# читаем изображения
import cv2
import PIL
from PIL import Image
from skimage import io

# работаем с изображениями
import numpy as np

import matplotlib.pyplot as plt

from skimage import data
from pathlib import Path

print("OpenCV:", cv2.__version__)
print("NumPy:", np.__version__)
print("PIL:", PIL.__version__)

In [ ]:
def show(*images, titles=None, cmap=None, figsize=(14, 5), vmin=None, vmax=None):
    """Удобный вывод одного или нескольких изображений."""
    n = len(images)
    titles = titles or [""] * n

    plt.figure(figsize=figsize)
    for i, (image, title) in enumerate(zip(images, titles), start=1):
        plt.subplot(1, n, i)
        plt.imshow(image, cmap=cmap, vmin=vmin, vmax=vmax)
        plt.title(title)
        plt.axis("off")
    plt.tight_layout()
    plt.show()

## 📷 Фотоэксперимент: «Что на самом деле снял телефон?»

Сделайте прямо сейчас **одну фотографию на телефон** и перенесите её на компьютер.

Лучше выбрать сцену, где есть одновременно:
- светлые и тёмные участки;
- мелкая текстура;
- яркий цвет;
- плавный фон.

Перед чтением файла попробуйте угадать:

1. Сколько пикселей в фотографии?
2. Какой будет `dtype` после чтения?
3. Где на гистограмме окажется больше значений?
4. Что сильнее пострадает при JPEG-сжатии: гладкий фон или мелкая текстура?

Если фото пока неудобно переносить, используйте `skimage.data.astronaut()` и вернитесь к эксперименту позже.

In [ ]:
# TODO: укажите путь к фотографии с телефона.
# Если фотографии пока нет, оставьте None и используйте astro ниже.
MY_PHOTO = None  # например: "my_photo.jpg"


In [ ]:
# TODO: прочитайте фотографию через OpenCV.
# 1. Проверьте, что путь существует.
# 2. Прочитайте файл с помощью cv2.imread.
# 3. Преобразуйте BGR -> RGB.
# 4. Выведите shape, dtype, min/max, size и само изображение.

if MY_PHOTO is not None and Path(MY_PHOTO).exists():
    phone_bgr = ...
    phone = ...

    print("shape:", ...)
    print("dtype:", ...)
    print("min/max:", ..., ...)
    print("Количество значений массива:", ...)

    show(phone, titles=["Наша фотография"])


**PIL / Pillow**

In [ ]:
# TODO: прочитайте ту же фотографию через PIL / Pillow,
# явно преобразуйте её в RGB и затем в NumPy-массив.

if MY_PHOTO is not None and Path(MY_PHOTO).exists():
    phone_pil = ...
    phone_pil = ...

    print("shape:", ...)
    print("dtype:", ...)
    print("min/max:", ..., ...)
    print("Количество значений массива:", ...)

    show(phone_pil, titles=["PIL"])


In [ ]:
# TODO: прочитайте ту же фотографию через scikit-image и изучите массив.

if MY_PHOTO is not None and Path(MY_PHOTO).exists():
    phone_skimage = ...

    print("shape:", ...)
    print("dtype:", ...)
    print("min/max:", ..., ...)
    print("Количество значений массива:", ...)

    show(phone_skimage, titles=["scikit-image"])


In [ ]:
# TODO: если вы загрузили собственное фото, сравните результаты трёх библиотек.
# Сначала предскажите: совпадут ли массивы после BGR -> RGB?

if MY_PHOTO is not None and Path(MY_PHOTO).exists():
    show(
        phone,
        phone_pil,
        phone_skimage,
        titles=["OpenCV → RGB", "PIL", "scikit-image"]
    )

    print("OpenCV == PIL:", ...)
    print("OpenCV == skimage:", ...)
    print("PIL == skimage:", ...)


## 1. Изображение = массив чисел

Возьмём стандартное RGB-изображение. В NumPy цветное изображение обычно имеет форму

$$ H \times W \times C $$

где `H` — высота, `W` — ширина, `C` — число каналов.

In [ ]:
astro = data.astronaut()  # RGB, uint8
print("shape:", astro.shape)
print("dtype:", astro.dtype)
print("min/max:", astro.min(), astro.max())
show(astro, titles=["Исходное RGB-изображение"])

### Задание 1.1 — порядок осей

Первая ось массива — **строки (y, высота)**, вторая — **столбцы (x, ширина)**.

Проверим это срезами.

In [ ]:
# TODO: с помощью срезов:
# 1. уберите верхние 100 строк;
# 2. оставьте левую половину изображения.
top_cut = ...
left_half = ...

show(
    top_cut, left_half,
    titles=["Без верхних 100 строк", "Левая половина"]
)

print("Исходная форма:", astro.shape)
print("После удаления верхних строк:", top_cut.shape)
print("После выделения левой половины:", left_half.shape)


## 2. RGB и BGR: одна из самых частых ошибок OpenCV

`matplotlib` и `scikit-image` работают с RGB.  
`OpenCV` традиционно использует BGR.

Посмотрим, что произойдёт, если передать RGB-массив в `cv2.imwrite` без преобразования.

### 🔮 Сначала предскажите

Представьте пиксель чистого красного цвета `RGB = [255, 0, 0]`.

OpenCV интерпретирует эти три числа как BGR. **Какой цвет получится, если забыть преобразование?**

Не запускайте код сразу. Сначала договоритесь об ответе и объясните его через порядок каналов.

In [ ]:
# TODO: сохраните RGB-массив через cv2.imwrite без перестановки каналов.
# Прочитайте файл обратно и приведите результат к RGB для показа.
cv2.imwrite("astro_wrong.png", ...)

back_wrong = ...
back_wrong_rgb = ...

show(
    astro, back_wrong_rgb,
    titles=["Исходник", "После неправильного cv2.imwrite"]
)

# TODO: вычислите MAE отдельно по каналам R, G, B.
diff_per_channel = ...
print("MAE по каналам [R, G, B]:", diff_per_channel)


In [ ]:
# TODO: сохраните изображение правильно:
# RGB -> BGR перед cv2.imwrite, затем прочитайте обратно и проверьте равенство.
astro_bgr = ...
cv2.imwrite("astro_ok.png", ...)

check = ...
check_rgb = ...

print("Совпадают:", ...)
show(astro, check_rgb, titles=["Исходник", "Корректно сохранённое изображение"])


## 3. Работа с пикселями и областями

Координаты массива задаются как `[y, x]`, а не `[x, y]`.

In [ ]:
# TODO: выберите пиксель по координатам [y, x], выведите RGB
# и закрасьте прямоугольную область красным цветом.
img = astro.copy()
y, x = 220, 260
print("RGB пикселя:", ...)

edited = img.copy()
edited[...] = ...

show(img, edited, titles=["Исходник", "Изменённая область"])


In [ ]:
# TODO: выделите R, G и B как три двумерных массива.
r = ...
g = ...
b = ...

show(r, g, b, titles=["R", "G", "B"], cmap="gray", vmin=0, vmax=255)


## 4. `uint8`, переполнение и переход к float

У `uint8` допустимы значения 0…255. Арифметика NumPy над `uint8` может привести к переполнению, поэтому для вычислений часто удобнее перейти к `float32` в диапазоне `[0, 1]`.

### 🔮 Предскажите без Python

Что получится?

```python
np.array([250], dtype=np.uint8) + np.uint8(20)
```

Варианты: `270`, `255`, `14`, ошибка.

Сначала выберите вариант и объясните, что должно произойти с 8 битами. Только потом запускайте следующую ячейку.

In [ ]:
# Сначала запишите предсказание в markdown-ячейке выше.
x = np.array([250], dtype=np.uint8)
print("uint8: 250 + 20 =", x + np.uint8(20))

# TODO: переведите astro в float32 и диапазон [0, 1].
astro_float = ...
print(astro_float.dtype, astro_float.min(), astro_float.max())

# TODO: увеличьте значения на 0.20, не выходя за границы [0, 1].
brighter = ...
show(astro_float, brighter, titles=["float [0,1]", "+0.20 к яркости"])


## 5. Crop и resize

У `cv2.resize` размер задаётся в порядке `(width, height)`, хотя `shape` хранится как `(height, width, channels)`. Это полезно запомнить.

In [ ]:
h, w = astro.shape[:2]

# TODO: вырежьте область [y: 80..400, x: 120..420].
crop = ...

# TODO: уменьшите ширину до 300, сохранив пропорции.
new_width = 300
scale = ...
new_height = ...

resized = cv2.resize(
    astro,
    (..., ...),
    interpolation=cv2.INTER_AREA
)

show(crop, resized, titles=["Crop", f"Resize: {resized.shape}"])
print("Исходник:", astro.shape)
print("Resize:", resized.shape)


In [ ]:
# TODO: сравните методы интерполяции при уменьшении изображения.
interpolations = {
    "NEAREST": cv2.INTER_NEAREST,
    "LINEAR": cv2.INTER_LINEAR,
    "CUBIC": cv2.INTER_CUBIC,
    "AREA": cv2.INTER_AREA,
    "LANCZOS4": cv2.INTER_LANCZOS4,
}

resized_images = []
titles = []

for name, interpolation in interpolations.items():
    img_resized = ...
    resized_images.append(img_resized)
    titles.append(name)

show(*resized_images, titles=titles)


In [ ]:
# TODO: повторите сравнение при сильном уменьшении до ширины 128.
small_width = 128
small_height = ...

resized_images = []
titles = []

for name, interpolation in interpolations.items():
    img_resized = ...
    resized_images.append(img_resized)
    titles.append(name)

show(*resized_images, titles=titles)


## 6. RGB → grayscale

Grayscale — не просто среднее трёх каналов. Вклад каналов различается из-за восприятия яркости.

In [ ]:
# TODO:
# 1. получите grayscale с помощью OpenCV;
# 2. получите grayscale простым средним трёх каналов;
# 3. сравните изображения и посчитайте MAE.
gray_cv = ...
gray_mean = ...

show(
    gray_cv, gray_mean,
    titles=["cv2.COLOR_RGB2GRAY", "Простое среднее RGB"],
    cmap="gray", vmin=0, vmax=255
)

mae = ...
print(f"MAE между способами: {mae:.2f}")


### 💥 Можно ли «сломать» grayscale?

Посмотрите вокруг и найдите **два предмета разных цветов**, которые, по вашему предположению, после перевода в grayscale будут иметь похожую яркость.

Сфотографируйте их вместе и проверьте гипотезу.

🧠 Обсудите: если два разных RGB-цвета превращаются почти в одно число, какую информацию мы потеряли? Можно ли восстановить исходный цвет только из grayscale?

## 7. Гистограмма яркости

Гистограмма показывает распределение значений интенсивности, но не хранит информацию о том, **где** находятся пиксели.

In [ ]:
# TODO: постройте гистограмму gray_cv: 256 интервалов, диапазон 0..256.
plt.figure(figsize=(9, 4))
...
plt.xlabel("Интенсивность")
plt.ylabel("Количество пикселей")
plt.title("Гистограмма grayscale-изображения")
plt.show()


### 🧠 Одинаковая гистограмма — одинаковое изображение?

Проверим важное ограничение гистограммы. Перемешаем все пиксели grayscale-изображения случайным образом.

**До запуска:** что произойдёт с картинкой? А с её гистограммой?

In [ ]:
rng = np.random.default_rng(42)

# TODO: перемешайте все значения gray_cv, не меняя их набор,
# и верните массиву исходную двумерную форму.
shuffled = ...
...
shuffled = ...

show(
    gray_cv, shuffled,
    titles=["Исходное изображение", "Те же пиксели в случайных местах"],
    cmap="gray", vmin=0, vmax=255
)

# TODO: наложите две гистограммы и проверьте равенство наборов значений.
fig, ax = plt.subplots(figsize=(9, 4))
...
ax.legend()
ax.set_title("Гистограммы")
plt.show()

print("Одинаковый набор значений:", ...)


**Вывод для обсуждения:** гистограмма знает, *сколько* пикселей каждой яркости существует, но не знает, *где* они расположены. Поэтому совершенно разные по структуре изображения могут иметь одну и ту же гистограмму.

## 8. Гамма и линейная яркость

Коды обычного изображения нельзя автоматически интерпретировать как количество света. Для учебного эксперимента используем приближение `gamma = 2.2`.

Сравним:
1. среднее кодов;
2. среднее после перехода в приближённо линейное пространство.

In [ ]:
black = np.zeros((120, 120), dtype=np.float32)
white = np.ones((120, 120), dtype=np.float32)

# TODO: найдите среднее кодированных значений.
mean_codes = ...

# Используем учебное приближение gamma = 2.2.
gamma = 2.2

# TODO:
# 1. декодируйте black и white в приближённо линейное пространство;
# 2. усредните линейный свет;
# 3. закодируйте среднее обратно.
black_linear = ...
white_linear = ...
mean_linear = ...
mean_linear_encoded = ...

show(
    mean_codes,
    mean_linear_encoded,
    titles=[
        f"Среднее кодов = {mean_codes[0,0]:.3f}",
        f"Среднее света → код = {mean_linear_encoded[0,0]:.3f}"
    ],
    cmap="gray", vmin=0, vmax=1
)

print("Среднее кодов:", mean_codes[0, 0])
print("Среднее в линейном пространстве, затем gamma:", mean_linear_encoded[0, 0])


## 🏆 JPEG challenge: «Сожми фото так, чтобы никто не заметил»

Используйте **свою фотографию** из начала занятия.

Правила для маленькой группы:

1. Каждый выбирает JPEG quality самостоятельно.
2. Цель — получить **как можно меньший файл**.
3. Остальные смотрят оригинал и сжатую версию **без знания quality**.
4. Если заметная деградация видна сразу — попытка не засчитывается.
5. После нескольких попыток сравните результаты.

Здесь нет единственного правильного `quality`.

🧠 После эксперимента обсудите:

- почему фотографии с мелкой текстурой и резкими границами сжимаются иначе, чем гладкие сцены;
- где артефакты появляются раньше;
- достаточно ли MAE, чтобы сказать, что изображение «выглядит так же»;
- почему повторное сохранение JPEG — отдельная проблема.

**Бонус:** найдите или сфотографируйте сцену, которую JPEG сжимает особенно хорошо, и сцену, которую он сжимает плохо.

## 9. JPEG — сжатие с потерями

Сохраним одно изображение с разным качеством JPEG и измерим ошибку относительно оригинала. PNG используем как контрольный lossless-формат.

In [ ]:
astro_bgr = cv2.cvtColor(astro, cv2.COLOR_RGB2BGR)

# TODO: сохраните JPEG с quality=95 и quality=20, а также PNG-контроль.
...
...
...

def read_rgb(path):
    # TODO: прочитайте файл через OpenCV и верните RGB.
    return ...

q95 = read_rgb("astro_q95.jpg")
q20 = read_rgb("astro_q20.jpg")
png = read_rgb("astro_lossless.png")

def mae(a, b):
    # TODO: реализуйте среднюю абсолютную ошибку без переполнения uint8.
    return ...

print("MAE PNG:", mae(astro, png))
print("MAE JPEG q95:", mae(astro, q95))
print("MAE JPEG q20:", mae(astro, q20))

print("PNG bytes:", Path("astro_lossless.png").stat().st_size)
print("JPEG q95 bytes:", Path("astro_q95.jpg").stat().st_size)
print("JPEG q20 bytes:", Path("astro_q20.jpg").stat().st_size)

show(astro, q95, q20, titles=["Original", "JPEG 95", "JPEG 20"])


## 10. Мини-задача на закрепление

Напишите функцию `prepare_image`, которая:

1. принимает RGB `uint8` изображение;
2. делает центральный квадратный crop;
3. изменяет размер до `size × size`;
4. переводит результат в grayscale;
5. возвращает `float32` в диапазоне `[0, 1]`.

Это типичный пример небольшого preprocessing pipeline.

In [ ]:
def prepare_image(image, size=224):
    """Центральный crop -> resize -> grayscale -> float32 [0, 1]."""
    # TODO: реализуйте полный preprocessing pipeline.
    pass


prepared = prepare_image(astro)

# После реализации раскомментируйте проверку:
# print(prepared.shape)
# print(prepared.dtype)
# print(prepared.min(), prepared.max())
# show(prepared, titles=["Результат preprocessing"], cmap="gray", vmin=0, vmax=1)


### Что важно вынести с семинара

- изображение в Python — прежде всего массив чисел;
- `shape` и координаты массива идут как `(H, W, C)` и `[y, x]`;
- RGB и BGR нельзя бездумно смешивать;
- `uint8` удобен для хранения, но опасен для арифметики;
- resize — это не просто изменение `shape`: нужен алгоритм интерполяции;
- grayscale и яркость имеют физический/перцептивный смысл;
- JPEG меняет значения пикселей, PNG — lossless;
- перед вычислениями важно понимать диапазон, dtype, цветовое пространство и происхождение пикселей.
